[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/11_optimizer_updates.ipynb)

# 11. Optimizer updates

모델 학습 대신 같은 작은 parameter와 gradient에 optimizer update를 직접 적용해 state와 ΔW를 비교한다.

**반복 형식:** 바닐라 PyTorch 실행 → profiler로 ATen/CUDA 연산 확인 → 필요할 때만 작은 텐서로 수학적 전개를 펼친다.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
print("torch:", torch.__version__)


In [ ]:
from torch.profiler import profile, ProfilerActivity

def profile_call(name, fn, *args, **kwargs):
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        out = fn(*args, **kwargs)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    print(f"\n[{name}] top operators")
    sort_key = "self_cuda_time_total" if torch.cuda.is_available() else "self_cpu_time_total"
    print(prof.key_averages().table(sort_by=sort_key, row_limit=12))

    return out


## 1. SGD

gradient 반대 방향으로 직접 이동한다.


In [ ]:
w0 = torch.tensor([[1., -1.], [0.5, 2.]], device=device)
g = torch.tensor([[0.2, -0.4], [1.0, 0.5]], device=device)
lr = 0.1

w_sgd = w0 - lr * g
print(w_sgd)


In [ ]:
_ = profile_call("SGD tensor update", lambda w, grad: w - lr * grad, w0, g)


## 2. Momentum

이전 gradient를 velocity에 누적한다.


In [ ]:
momentum = 0.9
v = torch.zeros_like(w0)

for step in range(3):
    v = momentum * v + g
    w0 = w0 - lr * v
    print("step", step, "velocity:\n", v, "\nweight:\n", w0)


In [ ]:
_ = profile_call("momentum update", lambda vel, grad: momentum * vel + grad, v, g)


## 3. RMSProp

gradient square EMA로 scale을 조절한다.


In [ ]:
w = torch.tensor([[1., -1.], [0.5, 2.]], device=device)
v2 = torch.zeros_like(w)
beta = 0.99

v2 = beta * v2 + (1 - beta) * g.square()
w = w - lr * g / (v2.sqrt() + 1e-8)

print("second moment:\n", v2)
print("weight:\n", w)


In [ ]:
_ = profile_call("RMSProp math", lambda grad: grad / ((1-beta)*grad.square()).sqrt().add(1e-8), g)


## 4. Adam / AdamW

1차·2차 EMA와 decoupled weight decay를 비교한다.


In [ ]:
w = torch.tensor([[1., -1.], [0.5, 2.]], device=device)
m = torch.zeros_like(w)
v = torch.zeros_like(w)
b1, b2 = 0.9, 0.999

m = b1 * m + (1 - b1) * g
v = b2 * v + (1 - b2) * g.square()

m_hat = m / (1 - b1)
v_hat = v / (1 - b2)

adam = w - lr * m_hat / (v_hat.sqrt() + 1e-8)
adamw = (1 - lr * 0.01) * w - lr * m_hat / (v_hat.sqrt() + 1e-8)

print("Adam:\n", adam)
print("AdamW:\n", adamw)


In [ ]:
_ = profile_call("Adam math", lambda: w - lr * m_hat / (v_hat.sqrt() + 1e-8))


## 5. Lion-style sign update

EMA 방향의 sign을 update에 사용한다.


In [ ]:
m = torch.zeros_like(g)
beta1, beta2 = 0.9, 0.99

update = (beta1 * m + (1 - beta1) * g).sign()
m = beta2 * m + (1 - beta2) * g

print("sign update:\n", update)
print("new momentum:\n", m)


In [ ]:
_ = profile_call("Lion sign", lambda grad: grad.sign(), g)


## 6. Prodigy-style D-adaptation idea

실제 전체 optimizer 구현 대신 adaptive global step scale D가 update 크기에 곱해지는 핵심을 분리해서 본다.


In [ ]:
base_lr = 1.0
D = torch.tensor(0.02, device=device)
normalized = g / (g.square().mean().sqrt() + 1e-8)
delta = -base_lr * D * normalized

print("D:", D.item())
print("adaptive delta:\n", delta)


In [ ]:
_ = profile_call("D-scaled update", lambda grad: -base_lr * D * grad / (grad.square().mean().sqrt() + 1e-8), g)


## 7. Muon-style Newton–Schulz orthogonalization

2D update matrix를 반복 matmul로 orthogonalized 방향에 가깝게 만든다.


In [ ]:
G = torch.tensor([[1., 2.], [3., 4.]], device=device)
X = G / (G.norm() + 1e-7)

for i in range(3):
    A = X @ X.T
    X = 1.5 * X - 0.5 * (A @ X)
    print("iteration", i, "\n", X)

print("singular values:", torch.linalg.svdvals(X))


In [ ]:
_ = profile_call("Newton-Schulz iteration", lambda z: 1.5 * z - 0.5 * ((z @ z.T) @ z), G / G.norm())


## References and provenance

**[11.1] SGD / Momentum**
- 출처: classical stochastic optimization
- 이 노트북에서 가져온 부분: baseline update

**[11.2] RMSProp / Adam / AdamW**
- 출처: Hinton lecture; Kingma & Ba; Loshchilov & Hutter
- 이 노트북에서 가져온 부분: adaptive moments and decoupled decay

**[11.3] Lion**
- 출처: Chen et al., Symbolic Discovery of Optimization Algorithms
- 이 노트북에서 가져온 부분: sign-based update

**[11.4] Prodigy**
- 출처: Mishchenko & Defazio, Prodigy
- 이 노트북에서 가져온 부분: D-adaptation learning-rate scaling

**[11.5] Muon**
- 출처: Muon original implementation/research; later large-model adoption including DeepSeek-related work
- 이 노트북에서 가져온 부분: matrix update orthogonalization via Newton–Schulz
